# RetailCast India — Data Audit (Section 1 of PROJECT_BLUEPRINT.md)

Reproduces, against the actual starter-kit CSVs, the checks that decide whether `market_signal.csv` and `vendor_signal.csv` are safe to use as horizon features, whether zero-inflation should be treated uniformly, and whether any festival falls inside the forecast horizon `d_1914`–`d_1941`. Every number below is computed live in this notebook — nothing is copied from the blueprint prose.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np
import pandas as pd
import data as data_mod

sales_wide = data_mod.load_sales_wide()
sales_long = data_mod.melt_sales_long(sales_wide)
cal = data_mod.load_calendar()
prices = data_mod.load_sell_prices()
mkt = data_mod.load_market_signal()
vendor = data_mod.load_vendor_signal()
print('sales_long', sales_long.shape, 'series:', sales_long['id'].nunique())
print('calendar', cal.shape)
print('market_signal', mkt.shape)
print('vendor_signal', vendor.shape)

sales_long (114780, 9) series: 60
calendar (1969, 15)
market_signal (114780, 3)
vendor_signal (116460, 3)


## 1.1 `market_signal.csv` — coverage window

In [2]:
mkt = mkt.copy()
mkt['d_num'] = mkt['d'].str.replace('d_', '', regex=False).astype(int)
vendor = vendor.copy()
vendor['d_num'] = vendor['d'].str.replace('d_', '', regex=False).astype(int)

print('market_signal d_num range:', mkt['d_num'].min(), '-', mkt['d_num'].max())
print('vendor_signal d_num range:', vendor['d_num'].min(), '-', vendor['d_num'].max())
print('sales history d_num range: 1 -', sales_long['d_num'].max())
print('forecast horizon: 1914 - 1941')
print()
print('market_signal covers the horizon (1914..1941)?', mkt['d_num'].max() >= 1941)
print('vendor_signal covers the horizon (1914..1941)?', vendor['d_num'].max() >= 1941)

market_signal d_num range: 1 - 1913
vendor_signal d_num range: 1 - 1941
sales history d_num range: 1 - 1913
forecast horizon: 1914 - 1941

market_signal covers the horizon (1914..1941)? False
vendor_signal covers the horizon (1914..1941)? True


## 1.1 / 1.2 Correlation with actual same-day sales, and scale ratio

In [3]:
def signal_vs_sales(signal_long, value_col, sales_long):
    joined = sales_long.merge(signal_long[['id', 'd_num', value_col]], on=['id', 'd_num'], how='inner')
    rows = []
    for sid, g in joined.groupby('id'):
        r = np.corrcoef(g['sales'], g[value_col])[0, 1] if g[value_col].std() > 0 else np.nan
        rows.append({
            'id': sid,
            'corr': r,
            'mean_sales': g['sales'].mean(),
            'mean_signal': g[value_col].mean(),
            'scale_ratio': g[value_col].mean() / g['sales'].mean() if g['sales'].mean() > 0 else np.nan,
        })
    return pd.DataFrame(rows)

mkt_stats = signal_vs_sales(mkt, 'mkt_signal', sales_long)
vendor_stats = signal_vs_sales(vendor, 'vendor_forecast', sales_long)

print('--- market_signal vs sales (history overlap only) ---')
print('correlation: min={:.2f} mean={:.2f} max={:.2f}'.format(mkt_stats['corr'].min(), mkt_stats['corr'].mean(), mkt_stats['corr'].max()))
print('scale ratio (signal/sales): min={:.2f} mean={:.2f} max={:.2f}'.format(mkt_stats['scale_ratio'].min(), mkt_stats['scale_ratio'].mean(), mkt_stats['scale_ratio'].max()))
print()
print('--- vendor_signal vs sales (full range incl. horizon window in history) ---')
print('correlation: min={:.2f} mean={:.2f} max={:.2f}'.format(vendor_stats['corr'].min(), vendor_stats['corr'].mean(), vendor_stats['corr'].max()))
print('scale ratio (forecast/sales): min={:.2f} mean={:.2f} max={:.2f}'.format(vendor_stats['scale_ratio'].min(), vendor_stats['scale_ratio'].mean(), vendor_stats['scale_ratio'].max()))

--- market_signal vs sales (history overlap only) ---
correlation: min=0.87 mean=0.92 max=0.96
scale ratio (signal/sales): min=9.76 mean=10.49 max=11.03

--- vendor_signal vs sales (full range incl. horizon window in history) ---
correlation: min=-0.00 mean=0.12 max=0.58
scale ratio (forecast/sales): min=0.99 mean=1.00 max=1.01


In [4]:
print('market_signal per-series correlation and scale detail:')
mkt_stats.sort_values('id').to_string(index=False)

market_signal per-series correlation and scale detail:


'                                   id     corr  mean_sales  mean_signal  scale_ratio\n  ELECTRONICS_1_CABLE_KA_1_validation 0.931792    0.667538     7.104077    10.642208\n  ELECTRONICS_1_CABLE_KA_2_validation 0.916966    0.903816     9.421223    10.423829\n  ELECTRONICS_1_CABLE_KA_3_validation 0.940900    0.529535     5.334658    10.074235\n  ELECTRONICS_1_CABLE_MH_1_validation 0.931948    0.769995     8.251228    10.715954\n  ELECTRONICS_1_CABLE_MH_2_validation 0.938675    0.438578     4.757554    10.847676\n  ELECTRONICS_1_CABLE_MH_3_validation 0.929484    0.960795    10.580659    11.012405\n  ELECTRONICS_1_CABLE_MH_4_validation 0.936547    0.498170     5.306377    10.651731\n  ELECTRONICS_1_CABLE_TN_1_validation 0.920711    0.467852     4.794041    10.246927\n  ELECTRONICS_1_CABLE_TN_2_validation 0.948698    0.294302     3.013853    10.240675\n  ELECTRONICS_1_CABLE_TN_3_validation 0.937732    0.316257     3.469890    10.971736\nELECTRONICS_1_CHARGER_KA_1_validation 0.913023    4.0

In [5]:
print('vendor_signal per-series correlation and scale detail:')
vendor_stats.sort_values('id').to_string(index=False)

vendor_signal per-series correlation and scale detail:


'                                   id      corr  mean_sales  mean_signal  scale_ratio\n  ELECTRONICS_1_CABLE_KA_1_validation  0.057284    0.667538     0.667308     0.999655\n  ELECTRONICS_1_CABLE_KA_2_validation  0.097678    0.903816     0.902603     0.998658\n  ELECTRONICS_1_CABLE_KA_3_validation  0.042802    0.529535     0.528024     0.997147\n  ELECTRONICS_1_CABLE_MH_1_validation  0.090629    0.769995     0.767585     0.996870\n  ELECTRONICS_1_CABLE_MH_2_validation  0.084399    0.438578     0.438092     0.998892\n  ELECTRONICS_1_CABLE_MH_3_validation  0.041396    0.960795     0.960946     1.000158\n  ELECTRONICS_1_CABLE_MH_4_validation  0.050638    0.498170     0.496184     0.996013\n  ELECTRONICS_1_CABLE_TN_1_validation  0.095580    0.467852     0.467784     0.999855\n  ELECTRONICS_1_CABLE_TN_2_validation  0.073276    0.294302     0.296890     1.008792\n  ELECTRONICS_1_CABLE_TN_3_validation  0.036447    0.316257     0.316911     1.002066\nELECTRONICS_1_CHARGER_KA_1_validation  0.0

**Verdict on `market_signal.csv`**: fill in after seeing the printed numbers above — expect near-perfect correlation and a sharp cutoff before the horizon (leakage-shaped). **Verdict on `vendor_signal.csv`**: expect full horizon coverage with weak, series-dependent correlation but a mean-matched scale (a legitimate but noisy vendor forecast product).

## 1.3 Zero-inflation / intermittent demand

In [6]:
zero_share = sales_long.groupby('id')['sales'].apply(lambda s: (s == 0).mean()).sort_values(ascending=False)
overall_zero_share = (sales_long['sales'] == 0).mean()
print(f'Overall zero-day share across all 60 series x {sales_long["d_num"].max()} days: {overall_zero_share:.1%}')
print(f'Series with >50% zero days: {(zero_share > 0.5).sum()} / {len(zero_share)}')
print()
print('Top 5 highest zero-share series:')
print(zero_share.head(5).to_string())
print()
print('Bottom 5 (lowest zero-share, i.e. densest demand) series:')
print(zero_share.tail(5).to_string())

Overall zero-day share across all 60 series x 1913 days: 40.1%
Series with >50% zero days: 24 / 60

Top 5 highest zero-share series:
id
HOMECARE_2_AGARBATTI_KA_3_validation    0.877156
HOMECARE_2_AGARBATTI_TN_2_validation    0.810768
ELECTRONICS_1_CABLE_TN_2_validation     0.804496
HOMECARE_2_AGARBATTI_MH_1_validation    0.784631
HOMECARE_2_AGARBATTI_KA_2_validation    0.778881

Bottom 5 (lowest zero-share, i.e. densest demand) series:
id
GROCERY_3_ATTA_TN_1_validation           0.089388
GROCERY_3_ATTA_MH_4_validation           0.087297
ELECTRONICS_1_CHARGER_MH_4_validation    0.082070
ELECTRONICS_1_CHARGER_MH_1_validation    0.069524
ELECTRONICS_1_CHARGER_MH_3_validation    0.061683


In [7]:
mean_sales = sales_long.groupby('id')['sales'].mean().sort_values(ascending=False)
print('Mean daily sales per series (highest 5, lowest 5):')
print(mean_sales.head(5).to_string())
print('...')
print(mean_sales.tail(5).to_string())

Mean daily sales per series (highest 5, lowest 5):
id
GROCERY_3_ATTA_MH_3_validation    134.912180
GROCERY_3_ATTA_MH_1_validation     66.775745
GROCERY_3_ATTA_TN_3_validation     62.464192
GROCERY_3_ATTA_KA_2_validation     62.074229
GROCERY_3_ATTA_KA_3_validation     57.031887
...
id
ELECTRONICS_1_CABLE_MH_2_validation     0.438578
HOMECARE_2_AGARBATTI_TN_2_validation    0.392054
ELECTRONICS_1_CABLE_TN_3_validation     0.316257
ELECTRONICS_1_CABLE_TN_2_validation     0.294302
HOMECARE_2_AGARBATTI_KA_3_validation    0.227392


## 1.3 Spike cross-reference against `calendar.csv` events (highest-volume series)

In [8]:
top_series = mean_sales.index[0]
g = sales_long[sales_long['id'] == top_series].merge(cal[['d_num', 'date', 'event_name_1', 'event_type_1']], on='d_num', how='left')
mean_s, std_s = g['sales'].mean(), g['sales'].std()
spikes = g[g['sales'] > mean_s + 3 * std_s].sort_values('sales', ascending=False)
print(f'Series: {top_series}  mean={mean_s:.1f}  max={g["sales"].max()}')
print(f'{len(spikes)} days > mean+3*std; event-labeled among them: {spikes["event_name_1"].notna().sum()}')
spikes[['date', 'sales', 'event_name_1', 'event_type_1']].head(15).to_string(index=False)

Series: GROCERY_3_ATTA_MH_3_validation  mean=134.9  max=612
23 days > mean+3*std; event-labeled among them: 4


'      date  sales     event_name_1 event_type_1\n2018-11-04    612           Diwali     National\n2020-06-27    555              NaN          NaN\n2020-08-22    541              NaN          NaN\n2019-07-06    523              NaN          NaN\n2019-07-27    518              NaN          NaN\n2020-08-08    515              NaN          NaN\n2018-12-22    511              NaN          NaN\n2018-12-20    504              NaN          NaN\n2019-09-08    499              NaN          NaN\n2019-09-28    496              NaN          NaN\n2019-09-07    493 Ganesh_Chaturthi    Religious\n2020-08-23    487              NaN          NaN\n2020-09-12    485              NaN          NaN\n2019-01-05    481              NaN          NaN\n2019-06-29    472              NaN          NaN'

## 1.3 Stockout-zero vs demand-zero: cross-reference against `sell_prices.csv`

In [9]:
panel = data_mod.build_panel(include_horizon=False)
zero_rows = panel[panel['sales'] == 0]
stockout_zero = (~zero_rows['has_price_row']).sum()
demand_zero = zero_rows['has_price_row'].sum()
print(f'Zero-sales rows with NO price row that week (stockout/not-sold zero): {stockout_zero:,}')
print(f'Zero-sales rows WITH a price row that week (real demand zero): {demand_zero:,}')
print(f'Share of all zero rows that are stockout-zeros: {stockout_zero / (stockout_zero + demand_zero):.1%}')

Zero-sales rows with NO price row that week (stockout/not-sold zero): 0
Zero-sales rows WITH a price row that week (real demand zero): 45,970
Share of all zero rows that are stockout-zeros: 0.0%


## 1.4 Festivals falling inside the forecast horizon `d_1914`–`d_1941`

In [10]:
horizon_cal = cal[(cal['d_num'] >= 1914) & (cal['d_num'] <= 1941)]
events_in_horizon = horizon_cal[horizon_cal['event_name_1'].notna() | horizon_cal['event_name_2'].notna()]
print(f'Horizon window: {horizon_cal["date"].min().date()} to {horizon_cal["date"].max().date()}')
print(f'{len(events_in_horizon)} event day(s) fall inside the 28-day forecast horizon:')
events_in_horizon[['date', 'd', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_MH', 'snap_KA', 'snap_TN']].to_string(index=False)

Horizon window: 2023-04-03 to 2023-04-30
2 event day(s) fall inside the 28-day forecast horizon:


'      date      d event_name_1 event_type_1  event_name_2  event_type_2  snap_MH  snap_KA  snap_TN\n2023-04-10 d_1921   Ram_Navami    Religious           NaN           NaN        1        1        0\n2023-04-17 d_1928  Eid_al_Fitr    Religious           NaN           NaN        0        1        1'

## Summary — verdicts to carry into `approach_summary.md`

Fill in from the executed output above (do not hand-copy blueprint numbers — these are this run's real numbers):
- `market_signal.csv` coverage cutoff, correlation range, scale ratio range → verdict.
- `vendor_signal.csv` coverage, correlation range, scale ratio range → verdict.
- Overall zero-share and count of series >50% zero.
- Stockout-zero vs demand-zero split.
- Any festival(s) inside `d_1914`–`d_1941` and which series/states they matter for.